# Pretrain IndPatchTST


In [2]:
import importlib.util
from pathlib import Path
import sys

# Fallback: if package not installed in this kernel, add repo root to sys.path
if importlib.util.find_spec('src') is None:
    ROOT = Path('..').resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))



**Prerequisite:** install the package in editable mode from the repo root:

```bash
pip install -e .
```


In [3]:
from pathlib import Path
import torch
import torch.nn as nn
from src.data.dataloader import build_etth1_dataloaders
from src.models.indpatchtst import build_model_from_config
from src.training.optuna_search import bayesian_search
from src.training.train_indpatchtst_reg import train_and_valid_loop

data_path = Path('../data/ETTh1.csv')
if not data_path.exists():
    raise FileNotFoundError('Place ETTh1.csv in ../data/')

WINDOW, HORIZON = 36, 24
train_dl, val_dl, n_features = build_etth1_dataloaders(
    str(data_path), window=WINDOW, horizon=HORIZON, batch_size=128
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recherche bayesienne (reduis n_trials pour un test rapide)
best_params, best_loss = bayesian_search(
    train_dl, val_dl, WINDOW, HORIZON, device, n_trials=10, max_epochs=5
)

# Re-entrainement long avec la meilleure config
model = build_model_from_config(best_params, n_features, WINDOW, HORIZON).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=best_params['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
criterion = nn.MSELoss()
train_and_valid_loop(
    model, train_dl, val_dl, opt, criterion, n_epochs=10, device=device, scheduler=scheduler
)


[I 2026-03-11 23:13:12,124] A new study created in memory with name: no-name-cfb0023f-2cf2-4bf5-a41c-7a1b88d487f8
c:\Users\elyan\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
[I 2026-03-11 23:13:30,396] Trial 0 finished with value: 0.24260219483370074 and parameters: {'d_model': 128, 'n_heads': 1, 'patch_len': 11, 'stride': 3, 'n_layers': 5, 'd_ff': 512, 'dropout': 0.10308477766956904, 'revin': False, 'lr': 0.00020298058052421552}. Best is trial 0 with value: 0.24260219483370074.
c:\Users\elyan\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
c:\Users\elyan\AppData\Local\Programs\Python\Python311\Lib\site-pa


=== Meilleure config bayésienne ===
{'d_model': 256, 'n_heads': 8, 'patch_len': 11, 'stride': 4, 'n_layers': 2, 'd_ff': 256, 'dropout': 0.0799663418334207, 'revin': False, 'lr': 0.00022150109033839915}
Meilleure valid loss: 0.2270
Epoch 01 | train=0.5242 | valid=0.4213
Epoch 02 | train=0.3736 | valid=0.2864
Epoch 03 | train=0.3070 | valid=0.2503
Epoch 04 | train=0.2720 | valid=0.2277
Epoch 05 | train=0.2407 | valid=0.1824
Epoch 06 | train=0.2153 | valid=0.2123
Epoch 07 | train=0.1955 | valid=0.1990
Epoch 08 | train=0.1812 | valid=0.2057
Epoch 09 | train=0.1708 | valid=0.2219
Epoch 10 | train=0.1623 | valid=0.1757


{'train_loss': [0.5241910922490375,
  0.3735891504212811,
  0.30700216288524323,
  0.27199794576586755,
  0.24069852802177374,
  0.21531066652624958,
  0.19552964140893472,
  0.18117063365237224,
  0.17080704365407492,
  0.16227349639153665],
 'valid_loss': [0.42132524819242695,
  0.2864463776692184,
  0.2503341157010721,
  0.2276768953326649,
  0.1823939344707778,
  0.2122801362304819,
  0.19895852292993724,
  0.20573561993586895,
  0.2219207490079028,
  0.17571350894483442]}